# Transaction Processing with Delta Lake Preparation

**Dataset:** a stream of credit-card transactions, `CreditCardNumber, Amount` (later also `ZipCode`), dropped as small CSV files by a simulated payment system.

### What you will learn
| # | Concept | What you will *see* in this notebook |
|---|---------|--------------------------------------|
| 1 | **Why a table format** | Plain files in a lake have no transactions. Delta adds a **transaction log** (`_delta_log/`) |
| 2 | **Streaming into Delta** | Each micro-batch becomes one atomic commit, so readers never see half a batch |
| 3 | **Schema enforcement & evolution** | A stream with a new column is rejected until we allow `mergeSchema` |
| 4 | **ACID DML** | `INSERT`, `UPDATE`, `DELETE`, `MERGE` on files in a lake |
| 5 | **Auditing & time travel** | `DESCRIBE HISTORY`, `versionAsOf`, `RESTORE` |
| 6 | **Streaming upserts (update mode)** | Running totals per card, written with `foreachBatch` + `MERGE` |

### Agenda
0. Setup: Spark + Delta Lake 4
1. A first Delta table and its transaction log
2. Loading data streams into Delta Lake
3. Reading Delta while streaming
4. Schema enforcement and schema evolution
5. A catalog table over Delta + ACID DML
6. Auditing and time travel
7. Streaming aggregations into Delta in **update** mode
8. Wrap-up
9. Exercises

## Part 0 — Setup: Spark + Delta Lake 4

Delta Lake is a library: a JAR for the JVM plus the `delta-spark` Python package. Each Delta release is compiled against one Spark version, and **the versions must match**:

| Delta Lake | Built for Spark |
|-----------|-----------------|
| 4.0.x | 4.0 |
| 4.1 – 4.3 | 4.1 |
| **4.4.x** (latest) | **4.2** |

With a mismatched Spark (for example 4.3), writes appear to work but reads, `UPDATE` and `MERGE` fail with `NoSuchMethodError`. The next cell installs the matching PySpark into this environment if needed. Run it **before** anything imports `pyspark`. If you change versions, restart the kernel.

In [1]:
import importlib.metadata as md, subprocess, sys

PYSPARK_VERSION, DELTA_VERSION = "4.2.0", "4.4.1"     # Delta 4.4.x is built for Spark 4.2

def installed(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

if installed("pyspark") != PYSPARK_VERSION or installed("delta-spark") != DELTA_VERSION:
    print(f"Installing pyspark=={PYSPARK_VERSION} and delta-spark=={DELTA_VERSION} (one time) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           f"pyspark=={PYSPARK_VERSION}", f"delta-spark=={DELTA_VERSION}"])
print("pyspark", installed("pyspark"), "| delta-spark", installed("delta-spark"))

Installing pyspark==4.2.0 and delta-spark==4.4.1 (one time) ...
pyspark 4.2.0 | delta-spark 4.4.1


In [2]:
import pyspark, os, sys, shutil, json, time, pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])

PROJECT_DIR = Path(os.getcwd())
LANDING_DIR = PROJECT_DIR / "landing" / "payments"        # files dropped by the payment system
DELTA_DIR   = PROJECT_DIR / "external-data" / "delta"     # our Delta tables
CKPT_DIR    = PROJECT_DIR / "tmp" / "delta_checkpoints"   # streaming checkpoints

builder = (SparkSession.builder
           .appName("transaction-processing-with-delta-lake-preparation")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
           .config("spark.ui.showConsoleProgress", "false"))          # keep notebook output readable
spark = configure_spark_with_delta_pip(builder).getOrCreate()   # adds io.delta:delta-spark_2.13 to spark.jars.packages
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")    # the default of 200 makes each micro-batch slow

print("Spark version :", spark.version)
print("Delta JARs    :", spark.conf.get("spark.jars.packages"))
spark

Python /usr/local/bin/python3.12
Java /usr/lib/jvm/java-21-openjdk-arm64


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-22926fa5-0e24-47ee-a5ea-3093192e45bf;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	foun

Spark version : 4.2.0
Delta JARs    : io.delta:delta-spark_4.2_2.13:4.4.1


In [3]:
pd.set_option("display.max_colwidth", 120)

def to_pandas(df, n=100):
    """Small results only: keeps NULLs as None and nested columns (maps, arrays) as Python objects."""
    rows = [r.asDict() for r in df.limit(n).collect()]
    return pd.DataFrame(rows, columns=df.columns).convert_dtypes()     # nullable ints show <NA>, not 30602.0

def sql(query, n=100):
    """Run Spark SQL and return the first n rows as a pandas DataFrame (pretty notebook display)."""
    return to_pandas(spark.sql(query), n)

def reset(*folders):
    """Empty the given folders so that each demo starts clean."""
    for folder in folders:
        shutil.rmtree(folder, ignore_errors=True)
        Path(folder).mkdir(parents=True)

def drop_batch(lines, folder=LANDING_DIR):
    """Simulate the payment system dropping one CSV file of transactions."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    n = len(list(folder.glob("*.csv"))) + 1
    path = folder / f"payments_{n:03d}.csv"
    path.write_text("\n".join(lines) + "\n")
    print(f"dropped {path.name}: {len(lines)} transaction(s)")

def history(path, n=20):
    """The most useful columns of a Delta table's history."""
    return to_pandas(DeltaTable.forPath(spark, str(path)).history(n)
                     .select("version", F.date_format("timestamp", "HH:mm:ss").alias("time"), "operation",
                             "operationParameters", "operationMetrics"))

def delta_log(path):
    """List the commit files in _delta_log."""
    for f in sorted((Path(path) / "_delta_log").glob("*.json")):
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

def tree(path, max_files=4):
    """Print a directory tree (hides .crc files). max_files=0 -> folders only."""
    path = Path(path)
    if not path.exists():
        print(f"{path}   <-- DOES NOT EXIST")
        return
    for root, dirs, files in os.walk(path):
        dirs.sort()
        depth = len(Path(root).relative_to(path).parts)
        indent = "    " * depth
        print(f"{indent}{Path(root).name}/")
        files = sorted(f for f in files if not f.startswith("."))
        for f in files[:max_files]:
            print(f"{indent}    {f}  ({(Path(root) / f).stat().st_size:,} bytes)")
        if max_files and len(files) > max_files:
            print(f"{indent}    ... {len(files) - max_files} more file(s)")

reset(LANDING_DIR, DELTA_DIR, CKPT_DIR)      # every run of the notebook starts from nothing

## Part 1 — A first Delta table and its transaction log

A Delta table is **Parquet files + a transaction log**. The log, `_delta_log/`, is a folder of numbered JSON files, one per commit. Each commit lists the data files it **adds** and **removes**. A reader builds the current table by replaying the log, so it only ever sees whole commits.

```
transactions_delta/
├── _delta_log/
│   ├── 00000000000000000000.json   <- version 0: add part-0001.parquet
│   └── 00000000000000000001.json   <- version 1: add part-0002.parquet
├── part-0001.parquet
└── part-0002.parquet
```

In [4]:
TX_SCHEMA = "CreditCardNumber LONG, Amount DOUBLE"
first_tx = spark.createDataFrame([(1111, 25.0), (2222, 120.5), (3333, 9.99)], TX_SCHEMA)

DEMO_DELTA = DELTA_DIR / "demo"
first_tx.write.format("delta").save(str(DEMO_DELTA))                                   # version 0
spark.createDataFrame([(1111, 4.5)], TX_SCHEMA).write.format("delta").mode("append").save(str(DEMO_DELTA))  # version 1

tree(DEMO_DELTA)

demo/
    part-00000-0c3c1647-5547-46ee-a664-7837cd6429e9-c000.snappy.parquet  (405 bytes)
    part-00000-2825068f-fbb9-42a6-89d3-79b15430b341-c000.snappy.parquet  (405 bytes)
    part-00003-ba324de0-a473-40e9-b54d-4f901620ffa0-c000.snappy.parquet  (782 bytes)
    part-00006-6387ff37-95a5-4b0f-b0f1-64466d1e8b08-c000.snappy.parquet  (781 bytes)
    ... 2 more file(s)
    _delta_log/
        00000000000000000000.crc  (3,077 bytes)
        00000000000000000000.json  (1,905 bytes)
        00000000000000000001.crc  (3,435 bytes)
        00000000000000000001.json  (734 bytes)
        _staged_commits/


In [5]:
# What a commit looks like: one JSON action per line
for line in (DEMO_DELTA / "_delta_log" / "00000000000000000001.json").read_text().splitlines():
    action = json.loads(line)
    kind = next(iter(action))
    detail = {k: action[kind][k] for k in ("path", "size", "dataChange", "operation") if k in action[kind]}
    print(f"{kind:<12} {detail}")

commitInfo   {'operation': 'WRITE'}
add          {'path': 'part-00009-c56ea38c-ea73-484f-a41c-2f25c98725be-c000.snappy.parquet', 'size': 782, 'dataChange': True}


In [9]:
history(str(DEMO_DELTA))

,version,time,operation,operationParameters,operationMetrics
0,1,01:11:36,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}","{'numOutputRows': '1', 'numOutputBytes': '1187', 'numFiles': '2'}"
1,0,01:11:35,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}","{'numOutputRows': '3', 'numOutputBytes': '2750', 'numFiles': '4'}"


In [6]:
spark.read.format("delta").load(str(DEMO_DELTA)).show()

+----------------+------+
|CreditCardNumber|Amount|
+----------------+------+
|            1111|  25.0|
|            1111|   4.5|
|            3333|  9.99|
|            2222| 120.5|
+----------------+------+



## Part 2 — Loading data streams into Delta Lake

The payment system drops CSV files into `landing/payments/`, one line per transaction: `CreditCardNumber,Amount`. We parse them and append them to a Delta table.

We use `trigger(availableNow=True)`: each run processes every file that arrived since the last run, then stops. The **checkpoint** remembers which files are done. This is how an hourly or nightly incremental load works, and it makes the notebook repeatable. The same code with `processingTime="5 seconds"` runs forever (Part 3).

In [7]:
TX_DELTA = DELTA_DIR / "transactions_delta"

def payments_stream(folder=LANDING_DIR, schema=TX_SCHEMA):
    """Streaming DataFrame of transactions from the landing folder (the schema is required for streams)."""
    return (spark.readStream
            .format("text").load(str(folder))
            .select(F.from_csv("value", schema, {"delimiter": ","}).alias("tx"))
            .select("tx.*"))

def load_new_payments(stream=None, target=TX_DELTA, checkpoint=CKPT_DIR / "load", **options):
    """One incremental run: append every new file to the Delta table, then stop."""
    q = ((stream if stream is not None else payments_stream()).writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", str(checkpoint))
         .options(**options)
         .trigger(availableNow=True)
         .start(str(target)))
    q.awaitTermination(20)
    return q

drop_batch(["1111,25.00", "2222,120.50", "3333,9.99"])
drop_batch(["1111,4.50", "4444,300.00"])
load_new_payments()
spark.read.format("delta").load(str(TX_DELTA)).orderBy("CreditCardNumber").show()

dropped payments_001.csv: 3 transaction(s)
dropped payments_002.csv: 2 transaction(s)
+----------------+------+
|CreditCardNumber|Amount|
+----------------+------+
|            1111|  25.0|
|            1111|   4.5|
|            2222| 120.5|
|            3333|  9.99|
|            4444| 300.0|
+----------------+------+



In [8]:
# Run it again with no new files: nothing is loaded twice
load_new_payments()
print("rows:", spark.read.format("delta").load(str(TX_DELTA)).count())

drop_batch(["2222,15.00", "5555,42.00"])
load_new_payments()
print("rows:", spark.read.format("delta").load(str(TX_DELTA)).count())
delta_log(TX_DELTA)
history(TX_DELTA)[["version", "operation", "operationParameters"]]

rows: 5
dropped payments_003.csv: 2 transaction(s)
rows: 7
  00000000000000000000.json  (1,715 bytes)
  00000000000000000001.json  (920 bytes)


,version,operation,operationParameters
0,1,STREAMING UPDATE,"{'epochId': '1', 'outputMode': 'Append', 'queryId': '3e9d7dfa-3127-4d0e-aeac-5f695ccaf99b'}"
1,0,STREAMING UPDATE,"{'epochId': '0', 'outputMode': 'Append', 'queryId': '3e9d7dfa-3127-4d0e-aeac-5f695ccaf99b'}"


Each run that had new data became **one commit** (`STREAMING UPDATE`, even though it is an append). The run with no files wrote nothing. The commit records the query id and epoch, so if a run crashes and restarts, Delta knows which batches are already in the table and does not write them twice. Streaming into Delta is **exactly-once**.

### 2.1 Optional: the same pipeline from a socket

This is the original Java example. It needs a socket server. In a terminal run `nc -lk 11111` (Windows: `ncat -lk 11111`), then type lines such as `7777,12.34`.

In [9]:
socket_tx = (spark.readStream
             .format("socket").option("host", STREAM_HOST).option("port", 11111).load()
             .select(F.from_csv("value", TX_SCHEMA, {"delimiter": ","}).alias("tx"))
             .select("tx.*"))
q_socket = (socket_tx.writeStream
            .format("delta")
            .option("checkpointLocation", str(CKPT_DIR / "socket"))
            .start(str(DELTA_DIR / "transactions_socket")))
q_socket.awaitTermination(30)
q_socket.stop()

In [10]:
if (DELTA_DIR / "transactions_socket" / "_delta_log").exists():
    spark.read.format("delta").load(str(DELTA_DIR / "transactions_socket")).show()
else:
    print("No socket data was received.")

+----------------+------+
|CreditCardNumber|Amount|
+----------------+------+
|            7777| 12.34|
|            7777|   1.0|
|            8888|   5.5|
+----------------+------+



## Part 3 — Reading Delta while streaming

Now the stream runs **continuously**, one micro-batch every 2 seconds. While it writes, we read the table as an analyst would. A reader always sees the **latest committed version**: whole batches, never a file that is half written.

> With a plain CSV or Parquet folder, a reader that lists the folder while a write is in progress can pick up partial output, or files from a failed job. The Delta log makes the switch from one version to the next atomic.

In [10]:
q_live = (payments_stream().writeStream
          .format("delta")
          .option("checkpointLocation", str(CKPT_DIR / "load"))      # same checkpoint: carries on from Part 2
          .trigger(processingTime="2 seconds")
          .start(str(TX_DELTA)))

for i in range(4):
    drop_batch([f"{6000 + i},{10.0 * (i + 1):.2f}", "1111,1.00"])
    time.sleep(3)
    snapshot = spark.read.format("delta").load(str(TX_DELTA))
    version = DeltaTable.forPath(spark, str(TX_DELTA)).history(1).first()["version"]
    print(f"   reader sees version {version}: {snapshot.count()} rows")

q_live.stop()

dropped payments_004.csv: 2 transaction(s)
   reader sees version 2: 9 rows
dropped payments_005.csv: 2 transaction(s)
   reader sees version 3: 11 rows
dropped payments_006.csv: 2 transaction(s)
   reader sees version 4: 13 rows
dropped payments_007.csv: 2 transaction(s)
   reader sees version 5: 15 rows


### 3.1 Delta as a streaming *source*

A Delta table can also feed another stream. This is how bronze → silver → gold pipelines are chained. The query below reads the transactions table as a stream and keeps only large payments.

In [11]:
LARGE_DELTA = DELTA_DIR / "large_transactions"
large = (spark.readStream.format("delta").load(str(TX_DELTA))
         .where("Amount >= 100")
         .withColumn("flagged_at", F.current_timestamp()))
load_new_payments(large, target=LARGE_DELTA, checkpoint=CKPT_DIR / "large")
spark.read.format("delta").load(str(LARGE_DELTA)).show(truncate=False)

+----------------+------+-----------------------+
|CreditCardNumber|Amount|flagged_at             |
+----------------+------+-----------------------+
|4444            |300.0 |2026-11-15 01:15:03.533|
|2222            |120.5 |2026-11-15 01:15:03.533|
+----------------+------+-----------------------+



## Part 4 — Schema enforcement and schema evolution

A second payment system sends a third column, `ZipCode`, and writes into the **same** table. Plain files would accept anything, and the next reader would discover the mess. Delta checks every write against the table's schema, which is stored in the log.

In [12]:
ZIP_LANDING = PROJECT_DIR / "landing" / "payments_with_zip"
ZIP_SCHEMA = "CreditCardNumber LONG, Amount DOUBLE, ZipCode LONG"
reset(ZIP_LANDING)
drop_batch(["1111,60.00,30602", "8888,19.99,30605"], folder=ZIP_LANDING)

try:
    load_new_payments(payments_stream(ZIP_LANDING, ZIP_SCHEMA), checkpoint=CKPT_DIR / "zip_rejected")
except Exception as e:
    print("Rejected ->", str(e).split("\n")[0][:300])

dropped payments_001.csv: 2 transaction(s)
Rejected -> [STREAM_FAILED] Query [id = 7cb9377d-2aeb-40f3-ae49-c374bda113b4, runId = bc1e0e72-e33d-4aec-abe9-7ab6d16e70ec] terminated with exception: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.


26/11/15 01:15:34 ERROR MicroBatchExecution: Query [id = 7cb9377d-2aeb-40f3-ae49-c374bda113b4, runId = bc1e0e72-e33d-4aec-abe9-7ab6d16e70ec] terminated with error
org.apache.spark.sql.delta.DeltaAnalysisExceptionWithSubErrors: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 94a25e1f-b7f0-47fb-b409-3ca3e7e442a6).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- CreditCardNumber: long (nullable = true)
 |-- Amount: double (nullable = true)


Data schema:
root
 |-- CreditCardNumber: long (nullable = true)
 |-- Amount: double (nullable = true)
 |-- ZipCode: long (nullable = true)


	at org.apache.spark.sql.delta.Meta

**Schema evolution** must be asked for explicitly. With `mergeSchema`, the new column is added to the table schema in the same commit. Old rows read it as `NULL`.

In [13]:
load_new_payments(payments_stream(ZIP_LANDING, ZIP_SCHEMA), checkpoint=CKPT_DIR / "zip", mergeSchema="true")

tx = spark.read.format("delta").load(str(TX_DELTA))
tx.printSchema()
tx.orderBy(F.col("ZipCode").desc_nulls_last()).show(5)

root
 |-- CreditCardNumber: long (nullable = true)
 |-- Amount: double (nullable = true)
 |-- ZipCode: long (nullable = true)

+----------------+------+-------+
|CreditCardNumber|Amount|ZipCode|
+----------------+------+-------+
|            8888| 19.99|  30605|
|            1111|  60.0|  30602|
|            6001|  20.0|   NULL|
|            1111|   1.0|   NULL|
|            1111|   4.5|   NULL|
+----------------+------+-------+
only showing top 5 rows


## Part 5 — A catalog table over Delta + ACID DML

Register the folder as a table so analysts can use SQL. No schema is given, because Delta keeps the schema **in the log**, not in the metastore. That is why the same table can be registered in any catalog (Hive, Glue, Unity Catalog) without copying anything.

> We use the in-memory catalog here, so the name lasts for this session only. Add `.enableHiveSupport()` to the builder to store it in `./metastore_db`, as in session 25.

In [14]:
spark.sql("DROP TABLE IF EXISTS transactions")
spark.sql(f"CREATE TABLE transactions USING DELTA LOCATION '{TX_DELTA.as_posix()}'")
sql("DESCRIBE DETAIL transactions")[["format", "location", "numFiles", "sizeInBytes", "minReaderVersion", "minWriterVersion"]]

,format,location,numFiles,sizeInBytes,minReaderVersion,minWriterVersion
0,delta,file:/user/work/data-engineering-preparation/external-data/delta/transactions_delta,8,6659,1,2


In [15]:
spark.sql("INSERT INTO transactions VALUES (6666, 6666, 666)")       # the row from the Java example
spark.sql("UPDATE transactions SET Amount = 66.66 WHERE CreditCardNumber = 6666 AND ZipCode = 666")
spark.sql("DELETE FROM transactions WHERE CreditCardNumber = 3333")  # e.g. a customer asked to be forgotten
sql("SELECT * FROM transactions ORDER BY CreditCardNumber")

,CreditCardNumber,Amount,ZipCode
0,1111,1.0,<NA>
1,1111,1.0,<NA>
2,1111,1.0,<NA>
3,1111,1.0,<NA>
4,1111,25.0,<NA>
5,1111,60.0,30602
6,1111,4.5,<NA>
7,2222,15.0,<NA>
8,2222,120.5,<NA>
9,4444,300.0,<NA>


`MERGE` is the workhorse of transaction processing: **upsert** a batch of changes in one atomic commit. Here a batch of chargebacks arrives. Known transactions are reversed, and unknown ones are recorded as new rows.

In [16]:
chargebacks = spark.createDataFrame([(2222, 120.5, None), (9999, 75.0, 30601)], "CreditCardNumber LONG, Amount DOUBLE, ZipCode LONG")

(DeltaTable.forName(spark, "transactions").alias("t")
    .merge(chargebacks.alias("c"), "t.CreditCardNumber = c.CreditCardNumber AND t.Amount = c.Amount")
    .whenMatchedUpdate(set={"Amount": "-t.Amount"})                     # reverse the original payment
    .whenNotMatchedInsertAll()
    .execute())

sql("SELECT * FROM transactions WHERE CreditCardNumber IN (2222, 9999) ORDER BY CreditCardNumber")

,CreditCardNumber,Amount,ZipCode
0,2222,15.0,<NA>
1,2222,-120.5,<NA>
2,9999,75.0,30601


## Part 6 — Auditing and time travel

Every commit stays in the log: who did what, when, and with which metrics. Because old data files are only *marked* as removed, earlier versions can still be read, until `VACUUM` deletes those files (by default, files older than 7 days).

In [17]:
history(TX_DELTA)[["version", "time", "operation", "operationMetrics"]]

,version,time,operation,operationMetrics
0,10,01:16:44,MERGE,"{'numOutputRows': '3', 'numTargetBytesAdded': '1067', 'numTargetRowsInserted': '1', 'numTargetRowsMatchedDeleted': '..."
1,9,01:16:42,DELETE,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1189', 'numDeletionVectorsRemoved': '0'..."
2,8,01:16:40,UPDATE,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1326', 'numDeletionVectorsRemoved': '0'..."
3,7,01:16:39,WRITE,"{'numOutputRows': '1', 'numOutputBytes': '1041', 'numFiles': '1'}"
4,6,01:16:03,STREAMING UPDATE,"{'numOutputRows': '2', 'numRemovedFiles': '0', 'numAddedFiles': '1', 'numOutputBytes': '1065'}"
5,5,01:14:48,STREAMING UPDATE,"{'numOutputRows': '2', 'numRemovedFiles': '0', 'numAddedFiles': '1', 'numOutputBytes': '798'}"
6,4,01:14:44,STREAMING UPDATE,"{'numOutputRows': '2', 'numRemovedFiles': '0', 'numAddedFiles': '1', 'numOutputBytes': '798'}"
7,3,01:14:40,STREAMING UPDATE,"{'numOutputRows': '2', 'numRemovedFiles': '0', 'numAddedFiles': '1', 'numOutputBytes': '797'}"
8,2,01:14:35,STREAMING UPDATE,"{'numOutputRows': '2', 'numRemovedFiles': '0', 'numAddedFiles': '1', 'numOutputBytes': '798'}"
9,1,01:13:44,STREAMING UPDATE,"{'numOutputRows': '2', 'numRemovedFiles': '0', 'numAddedFiles': '1', 'numOutputBytes': '798'}"


In [18]:
# Query an earlier version: the table as it was after the first load
v1 = spark.read.format("delta").option("versionAsOf", 1).load(str(TX_DELTA))
print("version 1 rows:", v1.count(), "| current rows:", spark.table("transactions").count())
v1.printSchema()            # no ZipCode yet: the schema is versioned too

version 1 rows: 7 | current rows: 18
root
 |-- CreditCardNumber: long (nullable = true)
 |-- Amount: double (nullable = true)



In [19]:
# The same in SQL, plus an audit question: which transactions were deleted?
delete_version = int(sql("DESCRIBE HISTORY transactions").query("operation == 'DELETE'")["version"].max())
sql(f"""
SELECT * FROM transactions VERSION AS OF {delete_version - 1}
EXCEPT
SELECT * FROM transactions VERSION AS OF {delete_version}
""")

,CreditCardNumber,Amount,ZipCode
0,3333,9.99,None


In [20]:
# Oops: the chargeback batch was a test file. Undo the MERGE.
# RESTORE is itself a new commit, so the undone version can still be read.
merge_version = int(sql("DESCRIBE HISTORY transactions").query("operation == 'MERGE'")["version"].max())
spark.sql(f"RESTORE TABLE transactions TO VERSION AS OF {merge_version - 1}")
history(TX_DELTA, 3)[["version", "operation", "operationParameters"]]

,version,operation,operationParameters
0,11,RESTORE,"{'version': '9', 'timestamp': None}"
1,10,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""((CreditCardNumber#13896L = CreditCardNumber#1389..."
2,9,DELETE,"{'predicate': '[""(CreditCardNumber#11772L = 3333)""]'}"


## Part 7 — Streaming aggregations into Delta in **update** mode

Goal: a table `card_balances` with one row per card, the **running total** spent and the number of transactions, kept up to date as payments stream in. This is a stateful aggregation, so recall the three output modes:

| Output mode | What each micro-batch emits | Delta sink? |
|------------|-----------------------------|-------------|
| `append` | only rows that will never change again | ✅ (needs a watermark for aggregations) |
| `complete` | the **whole** result table, every time | ✅ overwrites the table each batch |
| `update` | only the rows that **changed** in this batch | ❌ not directly, use `foreachBatch` + `MERGE` |

We use a fresh landing folder, so the totals start from zero.

In [22]:
SWIPES_LANDING = PROJECT_DIR / "landing" / "card_swipes"
reset(SWIPES_LANDING)

def card_totals():
    """Running totals per card: a stateful streaming aggregation."""
    return (payments_stream(SWIPES_LANDING)
            .groupBy("CreditCardNumber")
            .agg(F.round(F.sum("Amount"), 2).alias("total_amount"),
                 F.count("*").alias("num_transactions")))

drop_batch(["1111,10.00", "2222,20.00", "3333,30.00", "1111,5.00"], folder=SWIPES_LANDING)

dropped payments_001.csv: 4 transaction(s)


### 7.1 Update mode straight into the Delta sink is rejected

In [23]:
try:
    q = (card_totals().writeStream.format("delta").outputMode("update")
         .option("checkpointLocation", str(CKPT_DIR / "update_direct"))
         .trigger(availableNow=True).start(str(DELTA_DIR / "card_balances_direct")))
    q.awaitTermination()
except Exception as e:
    print("Rejected ->", str(e).split("\n")[0][:200])

Rejected -> [DELTA_UNSUPPORTED_OUTPUT_MODE] Data source org.apache.spark.sql.delta.sources.DeltaDataSource does not support Update output mode


Why? The Delta sink only **adds files**. An `update` batch says "card 1111 now has total 15", and that must *replace* the old row for card 1111, not sit next to it. Replacing rows by key is exactly what `MERGE` does.

### 7.2 `complete` mode works, but rewrites everything

For comparison, `complete` mode overwrites the whole table on every batch, even if only one card changed.

In [24]:
BALANCES_COMPLETE = DELTA_DIR / "card_balances_complete"

def run_complete():
    q = (card_totals().writeStream.format("delta").outputMode("complete")
         .option("checkpointLocation", str(CKPT_DIR / "complete"))
         .trigger(availableNow=True).start(str(BALANCES_COMPLETE)))
    q.awaitTermination()

run_complete()
spark.read.format("delta").load(str(BALANCES_COMPLETE)).orderBy("CreditCardNumber").show()

+----------------+------------+----------------+
|CreditCardNumber|total_amount|num_transactions|
+----------------+------------+----------------+
|            1111|        15.0|               2|
|            2222|        20.0|               1|
|            3333|        30.0|               1|
+----------------+------------+----------------+



In [30]:
spark.stop()